# EDA — H별 전처리 데이터셋 비교

`preprocess/data/processed/H{n}/{train,valid,test}.csv` 10개 horizon을 가로지르며 결측/라벨/분포/이상치/상관을 비교하고, 모델링에 쓸 H 2~3개 후보를 선정한다.

**산출물 5종**
1. H별 잔여 결측 요약표
2. H별 라벨/섹터 분포 요약표
3. 주요 피처 기술통계 비교표
4. H별 이상치 비율 top 컬럼 표 + clip 경계 몰림 점검
5. H별 고상관 피처쌍 표

## 1. 공통 설정 & 데이터 로드

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from eda.utils import (
    load_csv,
    classify_columns,
    missing_summary,
    high_missing_columns,
    plot_missing_heatmap,
    analyze_single_feature,
    compare_group_stats_by_label,
    plot_histogram,
    plot_boxplot_by_label,
    build_outlier_decision_table,
    plot_correlation_heatmap,
    get_high_corr_pairs,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

In [ ]:
BASE = PROJECT_ROOT / "preprocess" / "data" / "processed"
HS = ["H6", "H8", "H10", "H12", "H14", "H16", "H18", "H20", "H22", "H24"]
SPLITS = ["train", "valid", "test"]

datasets = {
    h: {s: load_csv(BASE / h / f"{s}.csv") for s in SPLITS}
    for h in HS
}

meta_by_h = {
    h: json.loads((BASE / h / "meta.json").read_text(encoding="utf-8"))
    for h in HS
}

# 행 수 sanity check
shape_rows = []
for h in HS:
    for s in SPLITS:
        df = datasets[h][s]
        shape_rows.append({
            "H": h,
            "split": s,
            "rows": len(df),
            "cols": df.shape[1],
            "pos": int((df["label"] == 1).sum()),
        })
shape_table = pd.DataFrame(shape_rows)
shape_table

In [ ]:
# 이후 단계에서 공용으로 쓸 수치 컬럼 리스트 (H 상관없이 동일하다고 가정)
cols_ref = classify_columns(datasets["H12"]["train"])
RATIO_COLS = cols_ref["ratio"]
RAW_VALUE_COLS = cols_ref["raw_value"]
print(f"ratio 컬럼 수: {len(RATIO_COLS)}")
print(f"raw_value 컬럼 수: {len(RAW_VALUE_COLS)}")

## 2. 결측치 잔여 점검

전처리 후 결측이 남는 컬럼이 있는지, 특정 H/split에만 몰리는지 확인한다. 우선 train 전체로 스캔하고, 잔여가 있는 H만 valid/test로 확장한다.

In [ ]:
# 산출물 ① — H별 잔여 결측 요약표 (train)
rows = []
leftover_by_h = {}
for h in HS:
    ms = missing_summary(datasets[h]["train"])
    leftover = ms[ms["null_count"] > 0]
    leftover_by_h[h] = leftover
    rows.append({
        "H": h,
        "n_missing_cols": len(leftover),
        "max_null_pct": float(leftover["null_pct"].max()) if len(leftover) else 0.0,
        "cols": leftover.index.tolist(),
    })
missing_table = pd.DataFrame(rows)
missing_table

In [ ]:
# 결측이 남은 H만 valid/test 확장 확인
problem_hs = [r["H"] for r in rows if r["n_missing_cols"] > 0]
print(f"잔여 결측 있는 H: {problem_hs}")

for h in problem_hs:
    for s in ["valid", "test"]:
        ms = missing_summary(datasets[h][s])
        leftover = ms[ms["null_count"] > 0]
        if len(leftover):
            print(f"\n[{h}/{s}]")
            print(leftover.head(10))

In [ ]:
# 필요 시 특정 H 히트맵 (예: 결측이 가장 이상한 H)
# plot_missing_heatmap(datasets["H12"]["train"])

## 3. 라벨 분포 & 섹터 쏠림

H가 커질수록 양성 비율이 어떻게 바뀌는지, split별/섹터별 쏠림이 있는지 확인한다.

In [ ]:
# 산출물 ② (a) — H × split 양성 비율
label_rows = []
for h in HS:
    for s in SPLITS:
        df = datasets[h][s]
        vc = df["label"].value_counts(normalize=True)
        label_rows.append({
            "H": h,
            "split": s,
            "n": len(df),
            "pos": int((df["label"] == 1).sum()),
            "pos_rate": round(float(vc.get(1, 0.0)), 5),
        })
label_table = pd.DataFrame(label_rows)
label_pivot = label_table.pivot(index="H", columns="split", values="pos_rate").loc[HS]
label_pivot

In [ ]:
# 양성 개수도 함께 보기 (valid/test 통계 의미성 확인)
pos_pivot = label_table.pivot(index="H", columns="split", values="pos").loc[HS]
pos_pivot

In [ ]:
# 산출물 ② (b) — H별 섹터별 양성 비율 (train)
sector_rows = []
for h in HS:
    tr = datasets[h]["train"]
    sr = tr.groupby("gics_sector")["label"].mean()
    sector_rows.append({"H": h, **{k: round(float(v), 5) for k, v in sr.items()}})
sector_table = pd.DataFrame(sector_rows).set_index("H").loc[HS]
sector_table

In [ ]:
# H별 섹터별 표본 수 (양성 비율이 0/1로 튀는 이유가 표본 부족인지 확인용)
sector_counts_rows = []
for h in HS:
    tr = datasets[h]["train"]
    sc = tr["gics_sector"].value_counts()
    sector_counts_rows.append({"H": h, **sc.to_dict()})
sector_counts = pd.DataFrame(sector_counts_rows).set_index("H").loc[HS]
sector_counts

## 4. 주요 피처 기술통계 & 분포

대표 피처 10개로 한정해 H별 요약통계와 label별 차이를 본다. 시각화는 H12/H18/H24만.

In [ ]:
TARGET_FEATURES = [
    "부채비율",
    "유동비율",
    "자기자본비율",
    "매출액순이익률",
    "자기자본순이익률",
    "총자본영업이익률",
    "총자본회전율",
    "차입금의존도",
    "매출총이익률",
    "현금비율",
]

# 컬럼 이름이 다를 가능성 체크
available = set(datasets["H12"]["train"].columns)
for f in TARGET_FEATURES:
    if f not in available:
        print(f"[MISSING] {f}")
TARGET_FEATURES = [f for f in TARGET_FEATURES if f in available]
print(f"\n사용할 피처: {TARGET_FEATURES}")

In [ ]:
# 산출물 ③ — 주요 피처 기술통계 비교표 (train)
stat_rows = []
for h in HS:
    df = datasets[h]["train"]
    for feat in TARGET_FEATURES:
        st = analyze_single_feature(df, feat)
        stat_rows.append({"H": h, "feature": feat, **st})
stats_table = pd.DataFrame(stat_rows)
stats_table.head(20)

In [ ]:
# 피처별 median이 H에 따라 어떻게 움직이는지 한눈에
median_pivot = stats_table.pivot(index="feature", columns="H", values="median")[HS]
median_pivot

In [ ]:
# 대표 H에 대해 healthy vs delisted 비교
REP_HS = ["H12", "H18", "H24"]
for h in REP_HS:
    print(f"\n=== {h} ===")
    df = datasets[h]["train"]
    for feat in TARGET_FEATURES[:5]:  # 5개만
        print(f"\n[{feat}]")
        print(compare_group_stats_by_label(df, feat).round(3))

In [ ]:
# 대표 H 시각화 — 피처별 히스토그램과 label boxplot
for h in REP_HS:
    df = datasets[h]["train"]
    for feat in ["부채비율", "자기자본비율", "매출액순이익률"]:
        if feat not in df.columns:
            continue
        print(f"--- {h} / {feat} ---")
        plot_histogram(df, feat)
        plot_boxplot_by_label(df, feat)

## 5. 이상치 & 클리핑 흔적

`build_outlier_decision_table`로 이상치 비율 높은 컬럼 top 10을 H별로 뽑고, `meta.json`의 `preprocessor.clip_bounds`와 실제 데이터 경계값을 비교해 clip 상·하한에 값이 몰리는 컬럼을 찾는다.

In [ ]:
# 산출물 ④ (a) — H별 outlier top 10
outlier_by_h = {}
for h in HS:
    tbl = build_outlier_decision_table(
        datasets[h]["train"],
        numeric_cols=RATIO_COLS,
    )
    outlier_by_h[h] = tbl

# 한 표로 합치기
top_rows = []
for h in HS:
    top = outlier_by_h[h].head(10)
    for _, r in top.iterrows():
        top_rows.append({
            "H": h,
            "column": r["column"],
            "outlier_ratio_1_5": r["outlier_ratio_1_5"],
            "extreme_ratio_3_0": r["extreme_ratio_3_0"],
        })
outlier_top_table = pd.DataFrame(top_rows)
outlier_top_table

In [ ]:
# 피처별 outlier_ratio_1_5이 H에 따라 어떻게 움직이는지
ratio_rows = []
for h in HS:
    tbl = outlier_by_h[h].set_index("column")["outlier_ratio_1_5"]
    ratio_rows.append({"H": h, **tbl.to_dict()})
outlier_ratio_pivot = pd.DataFrame(ratio_rows).set_index("H").loc[HS]
# 가장 이상치가 많은 컬럼 20개만
top_cols = outlier_ratio_pivot.mean(axis=0).sort_values(ascending=False).head(20).index
outlier_ratio_pivot[top_cols]

In [ ]:
# 산출물 ④ (b) — clip 경계 몰림 점검
# meta.json의 clip_bounds와 비교해서 lo/hi 값에 붙어 있는 행 비율 계산
EPS = 1e-6
clip_rows = []
for h in HS:
    bounds = meta_by_h[h].get("preprocessor", {}).get("clip_bounds", {})
    df = datasets[h]["train"]
    for col, (lo, hi) in bounds.items():
        if col not in df.columns:
            continue
        s = df[col].dropna()
        if len(s) == 0:
            continue
        lo_ratio = float(((s - lo).abs() < EPS).mean())
        hi_ratio = float(((s - hi).abs() < EPS).mean())
        clip_rows.append({
            "H": h,
            "column": col,
            "lo": round(lo, 4),
            "hi": round(hi, 4),
            "lo_ratio": round(lo_ratio, 4),
            "hi_ratio": round(hi_ratio, 4),
            "boundary_ratio": round(lo_ratio + hi_ratio, 4),
        })
clip_table = pd.DataFrame(clip_rows)
clip_table_sorted = clip_table.sort_values("boundary_ratio", ascending=False)
clip_table_sorted.head(30)

In [ ]:
# 컬럼 기준으로 H별 boundary_ratio 피벗 — 몰림이 심한 컬럼 식별
clip_pivot = clip_table.pivot(index="column", columns="H", values="boundary_ratio")
worst_cols = clip_pivot.max(axis=1).sort_values(ascending=False).head(15).index
clip_pivot.loc[worst_cols, HS]

## 6. 상관관계 / 다중공선성

train 기준으로 절대 상관 ≥ 0.8 쌍을 H별로 수집하고, 모든 H에서 반복되는 쌍을 drop 후보로 정리한다.

In [ ]:
# 산출물 ⑤ — H별 고상관 쌍
corr_rows = []
for h in HS:
    pairs = get_high_corr_pairs(datasets[h]["train"], threshold=0.8)
    for a, b, v in pairs:
        corr_rows.append({"H": h, "col_a": a, "col_b": b, "corr": v})
corr_table = pd.DataFrame(corr_rows)
print(f"고상관 쌍 총 {len(corr_table)}개 (H별 중복 포함)")
corr_table.head(20)

In [ ]:
# 모든 H에서 공통으로 등장하는 쌍 (= drop 후보)
pair_counts = (
    corr_table.groupby(["col_a", "col_b"])
    .agg(n_H=("H", "nunique"), mean_corr=("corr", "mean"))
    .sort_values(["n_H", "mean_corr"], ascending=[False, False])
)
common_high_corr = pair_counts[pair_counts["n_H"] >= len(HS) - 1]
common_high_corr

In [ ]:
# 대표 H만 상관 히트맵 (출력이 커서 원할 때만 실행)
# plot_correlation_heatmap(datasets["H12"]["train"])
# plot_correlation_heatmap(datasets["H24"]["train"])

## 7. 최종 H 후보 선정

위 5개 표를 근거로 H 후보를 뽑는다. 기준:
- 잔여 결측 거의 없음
- valid/test 양성 개수가 통계적으로 의미 있는 수준 (대략 20개 이상)
- 이상치/클립 경계 몰림이 극단적이지 않음
- 공통 고상관 쌍이 과하게 많지 않음

In [ ]:
# 한 화면 요약 — H별 스코어 카드
summary_rows = []
for h in HS:
    m = meta_by_h[h]
    missing_cols = len(leftover_by_h[h])
    outlier_mean = float(outlier_by_h[h]["outlier_ratio_1_5"].mean())
    clip_mean = float(clip_table[clip_table["H"] == h]["boundary_ratio"].mean()) if len(clip_table) else 0.0
    high_corr_n = int((corr_table["H"] == h).sum())
    summary_rows.append({
        "H": h,
        "train_pos": m.get("train_pos"),
        "valid_pos": m.get("valid_pos"),
        "test_pos": m.get("test_pos"),
        "imbalance": m.get("imbalance_ratio"),
        "missing_cols": missing_cols,
        "outlier_ratio_mean": round(outlier_mean, 4),
        "clip_boundary_mean": round(clip_mean, 4),
        "high_corr_pairs": high_corr_n,
    })
summary_table = pd.DataFrame(summary_rows).set_index("H").loc[HS]
summary_table

**관찰 메모 (채워 넣기)**

- 결측: ...
- 라벨/섹터: ...
- 기술통계: ...
- 이상치/클립: ...
- 상관/다중공선성: ...

**최종 H 후보**: `H__`, `H__`, `H__`